# 01 - Data exploration

The data is PhysioNet credentialed access and `data/` is git-ignored. Everything shown here is an
aggregate or a plot, and outputs are cleared before committing.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import data

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
FIGDIR = os.path.join("..", "outputs", "figures")

## Loading

`load_diary()` reads `hormones_and_selfreport.csv`. Step 2 is calendar only, so no wearable table
is touched.

In [ ]:
diary = data.load_diary()
print(f"rows: {len(diary):,}   columns: {diary.shape[1]}   participants: {diary.id.nunique()}")

## Columns and missingness

In [ ]:
data.missingness(diary)

Split by interval, because the two intervals do not carry the same fields.

In [ ]:
by_interval = pd.DataFrame({
    f"pct_missing_{int(iv)}": (100 * g.isna().mean()).round(2)
    for iv, g in diary.groupby("study_interval")
})
by_interval.loc[["lh", "estrogen", "pdg", "flow_volume", "flow_color", "phase"]]

`phase` is complete for Interval 2, so its period starts are taken from `phase == 'Menstrual'`.
That is a supplied label rather than a self-report.

In [ ]:
for iv, g in diary.groupby("study_interval"):
    print(f"Interval {int(iv)}: {g.id.nunique():>2} participants | "
          f"flow_volume on {g.flow_volume.notna().sum():>4} of {len(g):>4} days | "
          f"pdg on {g.pdg.notna().sum():>4} | phase on {g.phase.notna().sum():>4}")

## Coverage

There are no calendar dates, so the time axis is `day_in_study`. Interval 1 is days 1-90 and
Interval 2 restarts at 838.

In [ ]:
data.coverage_summary(diary)

The Interval 2 participants all returned from Interval 1, so a split has to group on `id`
across both.

In [ ]:
i1 = set(diary.loc[diary.study_interval == 2022, "id"])
i2 = set(diary.loc[diary.study_interval == 2024, "id"])
print(f"Interval 2 is a subset of Interval 1: {i2 <= i1}   overlap: {len(i1 & i2)}")

## Duplicate participant-days

A repeated day would count one bleeding report twice and could invent a period start.

In [ ]:
data.check_duplicates(diary)

Exclude spotting and very light bleeding from the period marker. `Light` and above count.

*Spotting is reported on about 200 days, more than `Heavy`.*

In [ ]:
flow = data.flow_value_counts(diary)
flow

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
sub = flow.dropna(subset=["flow_volume"]).sort_values("n_days")
ax.barh(sub.flow_volume, sub.n_days,
        color=["#c2185b" if b else "#b0bec5" for b in sub.counts_as_period_day])
ax.set_xlabel("participant-days reported")
ax.set_title("Reported flow. Pink counts as a period day", fontsize=10)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "flow_volume_distribution.png"), dpi=150, bbox_inches="tight")

The rule is tested in `tests/test_data.py`.

## Period starts and cycles

`period_starts()` finds the first day of each run of bleeding. `build_cycles()` pairs consecutive
starts. Each participant's last run has no next start, so it cannot be labelled and is dropped.

In [ ]:
starts = data.period_starts(diary, gap_days=3)
cycles = data.build_cycles(starts, min_len=15, max_len=60)
frame = data.make_daily_prediction_frame(cycles)
print(f"{len(cycles)} cycles from {cycles.id.nunique()} participants")
print(cycles.groupby("study_interval").size().to_string())
data.censoring_report(starts, cycles)

## Sensitivity to gap_days

`gap_days` decides whether a short break splits one period into two.

In [ ]:
rows = []
for g in range(1, 6):
    s = data.period_starts(diary, gap_days=g)
    c = data.build_cycles(s, min_len=15, max_len=60)
    rows.append({"gap_days": g, "period_starts": len(s), "usable_cycles": len(c),
                 "participants": c.id.nunique(),
                 "median_cycle_len": round(c.cycle_length.median(), 1)})
sensitivity = pd.DataFrame(rows)
sensitivity

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(sensitivity.gap_days, sensitivity.period_starts, "-o", label="period starts")
ax.plot(sensitivity.gap_days, sensitivity.usable_cycles, "-s", label="usable cycles")
ax.axvline(3, color="#c2185b", ls="--", lw=1, label="default")
ax.set_xlabel("gap_days"); ax.set_ylabel("count")
ax.set_title("Starts move with the threshold, usable cycles much less", fontsize=10)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "gap_days_sensitivity.png"), dpi=150, bbox_inches="tight")

The raw start count moves a lot but the usable cycle count moves much less, because a low
threshold produces short fragments that then fail the 15-day check.

## Cycle lengths

In [ ]:
desc = cycles.cycle_length.describe()[["count", "mean", "50%", "std", "min", "max"]]
desc.index = ["n_cycles", "mean", "median", "sd", "min", "max"]
desc.round(2).to_frame("cycle_length_days").T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].hist(cycles.cycle_length, bins=range(15, 62, 2), color="#2b6cb0", alpha=0.85)
axes[0].axvline(cycles.cycle_length.median(), color="#c2185b", ls="--",
                label=f"median {cycles.cycle_length.median():.0f} d")
axes[0].set_xlabel("cycle length (days)"); axes[0].set_ylabel("cycles")
axes[0].set_title("Completed cycle lengths", fontsize=10); axes[0].legend(fontsize=8)

within = cycles.groupby("id").cycle_length.std().dropna()
axes[1].hist(within, bins=12, color="#e07b39", alpha=0.85)
axes[1].set_xlabel("within-participant SD (days)"); axes[1].set_ylabel("participants")
axes[1].set_title("How consistent is each participant", fontsize=10)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "cycle_length_distributions.png"), dpi=150, bbox_inches="tight")
print(f"median within-participant SD: {within.median():.2f} days ({len(within)} participants with 2+ cycles)")

Within-participant SD is 2 to 3 days. That is the floor for a calendar-only model.

## Cycles per participant

In [ ]:
data.cycles_per_participant(cycles)

In [ ]:
per = cycles.groupby("id").size()
print(f"participants with a usable cycle: {len(per)}")
print(f"participants with 2+ cycles: {(per >= 2).sum()}")
print(f"total usable cycles: {len(cycles)}   median per participant: {per.median():.0f}")

## Leakage

`phase` labels Interval 2 only and is never a feature. `participant_split()` groups on `id`, so a
participant cannot appear on both sides. The last cycle of each participant has no observed end and
is dropped. `PersonalBaseline` reads cycles earlier than the one being predicted.

The check below confirms the target is reconstructable from `cycle_day` and `cycle_length` alone.

In [ ]:
holds = (frame.days_until_next_period == frame.cycle_length - frame.cycle_day + 1).all()
print(f"prediction rows: {len(frame):,}")
print(f"target range: {frame.days_until_next_period.min()} to {frame.days_until_next_period.max()} days")
print(f"cycle_day range: {frame.cycle_day.min()} to {frame.cycle_day.max()}   (resets every period)")
print(f"identity holds: {holds}")
print(f"columns a model can see: {[c for c in frame.columns if c != 'days_until_next_period']}")

## Summary

1. No calendar dates. `day_in_study` is the only time axis.
2. Interval 1 has bleeding reports and no `pdg`. Interval 2 has `pdg` and no bleeding reports, so
   its labels come from `phase`.
3. No duplicate participant-days.
4. Spotting is reported often enough that excluding it changes the label set.
5. Pooling both intervals gives 119 cycles from 39 participants.
6. `gap_days` moves the start count more than the usable cycle count.
7. Within-participant SD is 2 to 3 days, which caps any calendar-only model.
8. Most participants have two or three cycles.